# Quark XINT8 quantization for YOLO26n-cls person classifier

Takes a YOLO26n-cls FP32 PyTorch checkpoint (e.g. `models/phase1_25k_acc0.9424.pt`, test acc 90.00% / val 94.24%) and quantizes it to INT8 using AMD Quark, targeting Ryzen AI NPU deployment.

**Pipeline:**
1. Export FP32 PyTorch checkpoint → FP32 ONNX
2. Sanity-check the FP32 ONNX against the original PyTorch numbers (catches preprocessing bugs)
3. Build a calibration data reader on the held-out `calib/` split
4. Run Quark static PTQ with XINT8 spec + Cross-Layer Equalization (CLE)
5. Evaluate the quantized model on the held-out `test/` split
6. Compare FP32 vs INT8 — accuracy, per-class metrics, file size, CPU latency

**Locked-in decisions:**
- Config preset: **XINT8** (Ryzen AI tuned)
- Input shape: **fixed batch=4, imgsz=512** (Ryzen AI requires static shapes; matches deployment config)
- Calibration set: all 500 images from `datasets/person_or_not/calib/`
- Calibration method: **Entropy** (Quark default for vision)
- Test set: all of COCO val2017 with natural labels

**Heads up:** CPU latency benchmarks here use ONNX Runtime CPU EP. They confirm the quantized model runs correctly and give a rough sense of speedup vs FP32, but they do **not** represent actual Ryzen AI NPU latency. For real NPU numbers you need Ryzen AI hardware + the Vitis AI execution provider (see the final cell), or the IRON design in the parent directory.

The INT8 ONNX produced here is the artifact consumed by `../gen_yolo_data.py` and `../gen_yolo_silu_luts.py` to materialize per-block weight bins and SiLU LUTs for the IRON design.

Reference: [Quark ONNX YOLOv8 Ryzen AI tutorial](https://quark.docs.amd.com/latest/tutorials/onnx/ryzen_ai/yolov8/onnx_ryzen_ai_yolov8_tutorial.html)


In [ ]:
%pip install -q amd-quark onnxruntime onnx onnxslim

## Config + imports

In [ ]:
from pathlib import Path
from collections import defaultdict
import time

import numpy as np
from PIL import Image
import onnx
import onnxruntime as ort
from ultralytics import YOLO

# --- Config knobs ---
FP32_PT_PATH = Path("models/phase1_25k_acc0.9424.pt")
FP32_ONNX_PATH = Path("models/phase1_25k_fp32.onnx")
XINT8_ONNX_PATH = Path("models/phase1_25k_xint8.onnx")

TEST_DIR = Path("datasets/person_or_not/test")
CALIB_DIR = Path("datasets/person_or_not/calib")

IMGSZ = 512
BATCH = 4  # static; matches deployment batch and Ryzen AI fixed-shape requirement

# Class index → name (Ultralytics sorts subdir names alphabetically)
CLASS_NAMES = sorted([d.name for d in TEST_DIR.iterdir() if d.is_dir()])
PERSON_IDX = CLASS_NAMES.index("person")
print(f"Class names: {CLASS_NAMES}  (person index = {PERSON_IDX})")

assert FP32_PT_PATH.exists(), f"Missing {FP32_PT_PATH}"

## Step 1 — Export FP32 PyTorch → FP32 ONNX

Static shape (batch=4, 3, 512, 512). Ultralytics writes the .onnx alongside the .pt; we move it to a clearer name.

In [ ]:
import shutil

m = YOLO(str(FP32_PT_PATH))
exported_path = Path(
    m.export(format="onnx", imgsz=IMGSZ, batch=BATCH, dynamic=False, simplify=True)
)
print(f"Ultralytics exported to: {exported_path}")

if exported_path != FP32_ONNX_PATH:
    shutil.move(exported_path, FP32_ONNX_PATH)
    print(f"Moved to: {FP32_ONNX_PATH}")

# Inspect the ONNX input/output
graph = onnx.load(str(FP32_ONNX_PATH)).graph
for inp in graph.input:
    shape = [d.dim_value or d.dim_param for d in inp.type.tensor_type.shape.dim]
    print(f"  Input  '{inp.name}' shape: {shape}")
for out in graph.output:
    shape = [d.dim_value or d.dim_param for d in out.type.tensor_type.shape.dim]
    print(f"  Output '{out.name}' shape: {shape}")

print(f"\nFP32 ONNX size: {FP32_ONNX_PATH.stat().st_size / 1e6:.2f} MB")

## Step 2 — Reusable ONNX evaluation function

Preprocessing matches Ultralytics' classification pipeline: PIL → center-crop to square → resize to imgsz → `/255` → CHW. If our FP32 ONNX baseline numbers don't match the original PyTorch numbers (90.00% accuracy), this preprocessing is likely the culprit — Ultralytics may apply additional normalization (e.g., ImageNet mean/std) that we'd need to add here.

In [ ]:
def preprocess_for_onnx(img_path: Path, imgsz: int = IMGSZ) -> np.ndarray:
    """Ultralytics-cls-style preprocessing: center-crop, resize, /255, CHW. Returns (3, H, W) float32."""
    img = Image.open(img_path).convert("RGB")
    W, H = img.size
    side = min(W, H)
    left = (W - side) // 2
    top = (H - side) // 2
    img = img.crop((left, top, left + side, top + side)).resize(
        (imgsz, imgsz), Image.BILINEAR
    )
    arr = np.asarray(img, dtype=np.float32) / 255.0  # HWC in [0, 1]
    return arr.transpose(2, 0, 1)  # CHW


def evaluate_onnx_on_test(
    onnx_path: Path,
    test_dir: Path = TEST_DIR,
    imgsz: int = IMGSZ,
    batch: int = BATCH,
    providers=None,
):
    """Run an ONNX classifier on the test set. Returns accuracy, per-class metrics, per-image probs.

    Pads the final batch by repeating the last image to keep the fixed batch shape.
    """
    sess = ort.InferenceSession(
        str(onnx_path), providers=providers or ["CPUExecutionProvider"]
    )
    input_name = sess.get_inputs()[0].name

    class_names = sorted([d.name for d in test_dir.iterdir() if d.is_dir()])
    person_idx = class_names.index("person")
    confusion = defaultdict(int)
    records = []  # list of (filename, true_label, prob_person, pred_label)

    for cls_dir in sorted(test_dir.iterdir()):
        if not cls_dir.is_dir():
            continue
        true_label = cls_dir.name
        images = list(cls_dir.iterdir())
        for i in range(0, len(images), batch):
            chunk = images[i : i + batch]
            actual = len(chunk)
            # Pad batch by repeating last image so input shape stays static
            while len(chunk) < batch:
                chunk.append(chunk[-1])
            batch_tensor = np.stack(
                [preprocess_for_onnx(p, imgsz) for p in chunk]
            ).astype(np.float32)
            outs = sess.run(None, {input_name: batch_tensor})
            probs = outs[0]  # (batch, num_classes)
            for j in range(actual):
                p_person = float(probs[j, person_idx])
                pred_idx = int(np.argmax(probs[j]))
                pred_label = class_names[pred_idx]
                confusion[(true_label, pred_label)] += 1
                records.append((chunk[j].name, true_label, p_person, pred_label))

    n_total = sum(confusion.values())
    correct = sum(v for (t, p), v in confusion.items() if t == p)
    accuracy = correct / n_total

    print(f"  Accuracy: {accuracy:.4f}  ({correct}/{n_total})")
    per_class = {}
    for cls in class_names:
        tp = confusion[(cls, cls)]
        fp = sum(v for (t, p), v in confusion.items() if p == cls and t != cls)
        fn = sum(v for (t, p), v in confusion.items() if t == cls and p != cls)
        prec = tp / (tp + fp) if (tp + fp) else 0
        rec = tp / (tp + fn) if (tp + fn) else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
        per_class[cls] = {"precision": prec, "recall": rec, "f1": f1}
        print(f"  {cls:10s} prec={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}")

    return {
        "accuracy": accuracy,
        "per_class": per_class,
        "confusion": dict(confusion),
        "records": records,
    }

## Step 3 — FP32 ONNX baseline (sanity check vs PyTorch)

The PyTorch FP32 model scored **test accuracy = 90.00%** in the source training run. If the FP32 ONNX matches that within ±0.5%, our preprocessing is correct and we can trust subsequent comparisons. If it's substantially lower, the preprocessing function needs an additional normalization step (most likely ImageNet mean/std).


In [ ]:
print("=== FP32 ONNX on test set ===")
fp32_onnx_metrics = evaluate_onnx_on_test(FP32_ONNX_PATH)

delta = fp32_onnx_metrics["accuracy"] - 0.9000  # vs known PyTorch baseline
print(f"\nDelta vs PyTorch FP32 (90.00%): {delta:+.4f}")
if abs(delta) > 0.005:
    print("WARNING: FP32 ONNX accuracy differs from PyTorch FP32 by >0.5%.")
    print("  Preprocessing in preprocess_for_onnx() likely needs adjustment.")
    print("  Common fix: add ImageNet mean/std normalization after the /255 step.")
else:
    print("Preprocessing matches PyTorch within tolerance. Safe to proceed.")

## Step 4 — Calibration data reader

Quark's `quantize_model` consumes an `onnxruntime.quantization.CalibrationDataReader` that yields one feed dict per call (`{input_name: batch_tensor}`). We preload all 500 calib images into memory (small enough), pad the last batch, and serve them on demand.

The preprocessing here must **match the training/eval preprocessing exactly** — calibration should see inputs from the same distribution the deployed model will see.

In [ ]:
from onnxruntime.quantization import CalibrationDataReader


class CalibImageReader(CalibrationDataReader):
    """Serves preprocessed batches from datasets/person_or_not/calib/{person,no_person}."""

    def __init__(
        self, calib_dir: Path, model_path: Path, imgsz: int = IMGSZ, batch: int = BATCH
    ):
        sess = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
        self.input_name = sess.get_inputs()[0].name

        paths = []
        for cls_dir in sorted(calib_dir.iterdir()):
            if cls_dir.is_dir():
                paths.extend(sorted(cls_dir.iterdir()))
        print(f"Calibration set: {len(paths)} images from {calib_dir}")

        tensors = [preprocess_for_onnx(p, imgsz) for p in paths]
        self.batches = []
        for i in range(0, len(tensors), batch):
            grp = tensors[i : i + batch]
            while len(grp) < batch:
                grp.append(grp[-1])
            self.batches.append(np.stack(grp).astype(np.float32))
        print(f"Prepared {len(self.batches)} batches of {batch}")
        self.idx = 0

    def get_next(self):
        if self.idx >= len(self.batches):
            return None
        out = {self.input_name: self.batches[self.idx]}
        self.idx += 1
        return out

    def rewind(self):
        self.idx = 0

## Step 5 — Quark XINT8 static quantization

Uses the Ryzen AI tutorial's pattern: `XInt8Spec` for both activations and weights, plus `CLEConfig` (Cross-Layer Equalization — algorithmic improvement that rebalances weight magnitudes across layers before quantization, typically helps accuracy).

**TODO if accuracy collapses:** the tutorial referenced an `enable_npucnn=True` flag that doesn't appear in the public `QConfig` signature. If quantized accuracy is unexpectedly low, check Quark docs for the current way to opt into Ryzen AI's NPU-specific optimization passes.

In [ ]:
# Quark XINT8 + AdaRound, excluding the final Softmax from quantization.
#
# Empirically, quantizing the final Softmax causes a sign-flip on the
# logits that drops accuracy to ~44%. Keeping Softmax in FP32 (one
# exclude) keeps everything else INT8 and recovers ~89.5% accuracy —
# i.e. the rest of the network tolerates XINT8 fine; only the head
# Softmax was poisoning the output.
from quark.onnx import (
    XInt8Spec,
    AdaRoundConfig,
    ModelQuantizer,
    QConfig,
    QLayerConfig,
)

EXCLUDE_FINAL_HEAD = [
    "/model.10/Softmax",  # only the final output activation
]

quant_config = QConfig(
    global_config=QLayerConfig(activation=XInt8Spec(), weight=XInt8Spec()),
    algo_config=[AdaRoundConfig()],
    exclude=EXCLUDE_FINAL_HEAD,
    use_external_data_format=False,
)
print(
    f"Excluding {len(EXCLUDE_FINAL_HEAD)} node (Softmax only) from XINT8 quantization."
)

calib_reader = CalibImageReader(CALIB_DIR, FP32_ONNX_PATH)

print("\nRunning Quark XINT8+AdaRound with Softmax excluded...")
quantizer = ModelQuantizer(quant_config)
quantizer.quantize_model(
    str(FP32_ONNX_PATH),
    str(XINT8_ONNX_PATH),
    calib_reader,
)

fp32_size = FP32_ONNX_PATH.stat().st_size / 1e6
xint8_size = XINT8_ONNX_PATH.stat().st_size / 1e6
print(f"\nQuantization done.")
print(f"  FP32 ONNX:  {fp32_size:.2f} MB")
print(f"  XINT8 ONNX: {xint8_size:.2f} MB  ({xint8_size / fp32_size:.2f}× of FP32)")

## Step 6 — Evaluate XINT8 ONNX on test set

In [ ]:
print("=== XINT8 ONNX on test set ===")
xint8_metrics = evaluate_onnx_on_test(XINT8_ONNX_PATH)

# Threshold sweep + ROC AUC for the quantized model.
from sklearn.metrics import roc_auc_score, average_precision_score

y_true_xint8 = np.array(
    [1 if r[1] == "person" else 0 for r in xint8_metrics["records"]]
)
y_score_xint8 = np.array([r[2] for r in xint8_metrics["records"]])

print(f"\nROC AUC:           {roc_auc_score(y_true_xint8, y_score_xint8):.4f}")
print(f"Average Precision: {average_precision_score(y_true_xint8, y_score_xint8):.4f}")

print("\nThreshold sweep (predict 'person' if prob_person >= threshold):")
print(
    f"  {'thr':>5s} {'recall':>8s} {'FPR':>8s} {'precision':>10s} {'F1':>7s} {'accuracy':>10s}"
)
for t in [0.10, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    y_pred = (y_score_xint8 >= t).astype(int)
    tp = int(((y_pred == 1) & (y_true_xint8 == 1)).sum())
    fp = int(((y_pred == 1) & (y_true_xint8 == 0)).sum())
    tn = int(((y_pred == 0) & (y_true_xint8 == 0)).sum())
    fn = int(((y_pred == 0) & (y_true_xint8 == 1)).sum())
    rec = tp / (tp + fn) if (tp + fn) else 0
    fpr = fp / (fp + tn) if (fp + tn) else 0
    prec = tp / (tp + fp) if (tp + fp) else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
    acc = (tp + tn) / len(y_true_xint8)
    print(f"  {t:>5.2f} {rec:>8.4f} {fpr:>8.4f} {prec:>10.4f} {f1:>7.4f} {acc:>10.4f}")

## Step 7 — FP32 vs XINT8 comparison

In [ ]:
import pandas as pd

rows = []
for label, m in (
    (
        "FP32 PyTorch (training baseline)",
        {
            "accuracy": 0.9000,
            "per_class": {
                "person": {"precision": 0.9782, "recall": 0.8329, "f1": 0.8997},
                "no_person": {"precision": 0.8338, "recall": 0.9783, "f1": 0.9003},
            },
        },
    ),
    ("FP32 ONNX", fp32_onnx_metrics),
    ("XINT8 ONNX", xint8_metrics),
):
    rows.append(
        {
            "model": label,
            "accuracy": m["accuracy"],
            "person_precision": m["per_class"]["person"]["precision"],
            "person_recall": m["per_class"]["person"]["recall"],
            "person_f1": m["per_class"]["person"]["f1"],
            "no_person_precision": m["per_class"]["no_person"]["precision"],
            "no_person_recall": m["per_class"]["no_person"]["recall"],
        }
    )
df = pd.DataFrame(rows)
df

## Step 8 — CPU latency benchmark (FP32 ONNX vs XINT8 ONNX)

ONNX Runtime CPU EP, fixed batch=4, imgsz=512. Reports per-batch and per-image latency. **Not a Ryzen AI NPU measurement** — see the deployment notes below.

In [ ]:
def bench_onnx(
    onnx_path: Path, n_batches: int = 20, batch: int = BATCH, imgsz: int = IMGSZ
):
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    input_name = sess.get_inputs()[0].name
    sample = next(next(d for d in sorted(TEST_DIR.iterdir()) if d.is_dir()).iterdir())
    batch_tensor = np.stack([preprocess_for_onnx(sample, imgsz)] * batch).astype(
        np.float32
    )

    for _ in range(3):  # warm up
        sess.run(None, {input_name: batch_tensor})

    times = []
    for _ in range(n_batches):
        t0 = time.perf_counter()
        sess.run(None, {input_name: batch_tensor})
        times.append((time.perf_counter() - t0) * 1000)
    return np.array(times)


def report_latency(label: str, times_ms: np.ndarray, batch: int = BATCH):
    mean_batch = times_ms.mean()
    p50_batch = np.percentile(times_ms, 50)
    p95_batch = np.percentile(times_ms, 95)
    mean_img = mean_batch / batch
    p50_img = p50_batch / batch
    p95_img = p95_batch / batch
    batch_fps = 1000.0 / mean_batch  # batches/sec (1 batch = 4 images)
    image_fps = 1000.0 * batch / mean_batch  # images/sec  (== 4 × batch_fps)
    print(f"{label}:")
    print(
        f"  per-batch:   mean={mean_batch:6.1f} ms   p50={p50_batch:6.1f} ms   p95={p95_batch:6.1f} ms"
    )
    print(
        f"  per-image:   mean={mean_img:6.1f} ms   p50={p50_img:6.1f} ms   p95={p95_img:6.1f} ms"
    )
    print(f"  throughput:  {batch_fps:6.1f} batches/sec   {image_fps:6.1f} images/sec")


t_fp32 = bench_onnx(FP32_ONNX_PATH)
report_latency(f"FP32 ONNX  (batch={BATCH}, imgsz={IMGSZ})", t_fp32)
print()
t_xint8 = bench_onnx(XINT8_ONNX_PATH)
report_latency(f"XINT8 ONNX (batch={BATCH}, imgsz={IMGSZ})", t_xint8)

speedup = t_fp32.mean() / t_xint8.mean()
fps_gain = (1000.0 * BATCH / t_xint8.mean()) - (1000.0 * BATCH / t_fp32.mean())
print(f"\nSpeedup (FP32 → XINT8): {speedup:.2f}×   (+{fps_gain:.1f} images/sec)")

## Notes for actual Ryzen AI NPU deployment

What this notebook produced:
- `models/phase1_25k_fp32.onnx` — FP32 reference
- `models/phase1_25k_xint8.onnx` — INT8 quantized, deployment artifact for Ryzen AI

**To run on Ryzen AI via ONNX Runtime + Vitis AI EP:**
1. Install the [Ryzen AI Software Stack](https://ryzenai.docs.amd.com/) on a supported Ryzen AI laptop/CPU
2. Install the Vitis AI Execution Provider for ONNX Runtime
3. Create the InferenceSession with the Vitis AI EP:
    ```python
    sess = ort.InferenceSession(
        "models/phase1_25k_xint8.onnx",
        providers=["VitisAIExecutionProvider"],
        provider_options=[{"config_file": "/path/to/vaip_config.json"}],
    )
    ```
4. Benchmark there — that's the number that matters for deployment latency.

**To run on Ryzen AI via the IRON design in the parent directory:**
The parent directory contains an end-to-end IRON design that compiles this XINT8 ONNX into per-block kernels and a chained xclbin running on the Strix Point NPU2 array. See the parent `README.md` for the full recipe (`make data`, `make chain`, `make run_chain`).

**If the quantized model doesn't compile to the NPU**: check Quark's [Ryzen AI best practice](https://quark.docs.amd.com/latest/supported_accelerators/ryzenai/ryzen_ai_best_practice.html) page — it covers op support, custom calibration tweaks, and the auto-search workflow that finds quantization configs matching the NPU's op set.

**Other things worth trying after this works:**
- Quark's auto-search to find a better config than vanilla XINT8
- AdaQuant fine-tuning if XINT8 accuracy is unacceptable
- A8W8 as a portable comparison (runs on any CPU; useful if you ever deploy to non-AMD hardware)
